# GCN Cloud Notebook: Richer Graph Features + Target Normalization

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and runs the current best single-run GCN workflow with target normalization.

This standalone notebook combines the positive improvements confirmed so far:
- Adam with the baseline two-phase schedule
- weighted edges from `adjacency_area.csv`
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling
- standardized regression targets during training

No new lattice sets are required.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/GCN_Cloud_Outputs_Richer_Graph_Features_Target_Normalized'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/GCN_Cloud_Outputs_Richer_Graph_Features_Target_Normalized'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs_richer_graph_features_target_normalized') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs_richer_graph_features_target_normalized'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
from copy import deepcopy
import json
import numpy as np
import pandas as pd
import shutil
import subprocess
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.preprocessing import StandardScaler

from colab_gnn_stiffness_prototype import (
    TrainingConfig,
    SimpleGNN,
    create_data_loaders,
    compute_regression_metrics,
    load_lattice_dataset,
    normalize_feature_splits,
    plot_prediction_splits,
    plot_training_history,
    save_run_artifacts,
    set_seed,
    split_dataset,
    summarize_metrics,
)


def normalize_target_splits(train_data, val_data, test_data):
    train_targets = np.asarray([float(sample.y.item()) for sample in train_data], dtype=np.float32).reshape(-1, 1)
    target_scaler = StandardScaler().fit(train_targets)

    for split in (train_data, val_data, test_data):
        for sample in split:
            scaled_target = target_scaler.transform(np.asarray([[float(sample.y.item())]], dtype=np.float32)).astype(np.float32)
            sample.y = torch.from_numpy(scaled_target.reshape(-1))

    return target_scaler


def transform_prediction_split_targets(prediction_data, target_scaler):
    for sample in prediction_data:
        scaled_target = target_scaler.transform(np.asarray([[float(sample.y.item())]], dtype=np.float32)).astype(np.float32)
        sample.y = torch.from_numpy(scaled_target.reshape(-1))


def inverse_transform_vector(values: np.ndarray, target_scaler: StandardScaler) -> np.ndarray:
    return target_scaler.inverse_transform(values.reshape(-1, 1)).reshape(-1).astype(np.float32)


def run_epoch_target_normalized(model, data_loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(mode=training)

    total_loss = 0.0
    for batch in data_loader:
        batch = batch.to(device)
        if training:
            optimizer.zero_grad()

        predictions = model(batch)
        loss = criterion(predictions, batch.y.view(-1, 1))

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(data_loader.dataset)


def train_target_normalized_model(model, train_loader, val_loader, config):
    model.to(config.device)
    criterion = nn.MSELoss()

    history = {
        'train_losses': [],
        'val_losses': [],
        'best_val_loss': float('inf'),
        'epochs_completed': 0,
    }
    best_state = deepcopy(model.state_dict())

    phases = (
        ('phase_1', config.lr_phase1, config.epochs_phase1),
        ('phase_2', config.lr_phase2, config.epochs_phase2),
    )

    epochs_completed = 0
    for phase_name, learning_rate, phase_epochs in phases:
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=config.weight_decay)
        stale_epochs = 0

        print(f'Starting {phase_name}: lr={learning_rate}, epochs={phase_epochs}')
        for _ in range(phase_epochs):
            train_loss = run_epoch_target_normalized(model, train_loader, criterion, config.device, optimizer=optimizer)
            val_loss = run_epoch_target_normalized(model, val_loader, criterion, config.device)

            history['train_losses'].append(train_loss)
            history['val_losses'].append(val_loss)
            epochs_completed += 1

            if val_loss < history['best_val_loss']:
                history['best_val_loss'] = val_loss
                best_state = deepcopy(model.state_dict())
                stale_epochs = 0
            else:
                stale_epochs += 1

            if epochs_completed == 1 or epochs_completed % 20 == 0:
                print(f'  epoch {epochs_completed:>4}/{config.total_epochs} train={train_loss:.6f} val={val_loss:.6f}')

            if stale_epochs >= config.patience:
                print(f'  early stopping triggered during {phase_name}')
                break

    model.load_state_dict(best_state)
    history['epochs_completed'] = epochs_completed
    return history


def evaluate_target_normalized_model(model, data_loader, target_scaler, device='cpu'):
    model.eval()
    model.to(device)

    scaled_predictions = []
    scaled_ground_truth = []
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            batch_predictions = model(batch).cpu().numpy().ravel()
            scaled_predictions.extend(batch_predictions.tolist())
            scaled_ground_truth.extend(batch.y.cpu().numpy().ravel().tolist())

    scaled_prediction_array = np.asarray(scaled_predictions, dtype=np.float32)
    scaled_ground_truth_array = np.asarray(scaled_ground_truth, dtype=np.float32)
    prediction_array = inverse_transform_vector(scaled_prediction_array, target_scaler)
    ground_truth_array = inverse_transform_vector(scaled_ground_truth_array, target_scaler)
    metrics = compute_regression_metrics(prediction_array, ground_truth_array)
    return prediction_array, ground_truth_array, metrics


def predict_on_directory_target_normalized(model, data_root, feature_scaler, target_scaler, batch_size=1, device='cpu'):
    prediction_data = load_lattice_dataset(data_root)
    for sample in prediction_data:
        transformed = feature_scaler.transform(sample.x.numpy()).astype(np.float32)
        sample.x = torch.from_numpy(transformed)
        if hasattr(feature_scaler, 'graph_mean_') and hasattr(feature_scaler, 'graph_scale_'):
            graph_mean = np.asarray(feature_scaler.graph_mean_, dtype=np.float32)
            graph_scale = np.asarray(feature_scaler.graph_scale_, dtype=np.float32)
            graph_scale = np.where(graph_scale == 0.0, 1.0, graph_scale)
            graph_transformed = ((sample.graph_attr.numpy() - graph_mean) / graph_scale).astype(np.float32)
            sample.graph_attr = torch.from_numpy(graph_transformed)
    transform_prediction_split_targets(prediction_data, target_scaler)

    from torch_geometric.loader import DataLoader
    loader = DataLoader(prediction_data, batch_size=batch_size, shuffle=False)
    predictions, ground_truth, metrics = evaluate_target_normalized_model(model, loader, target_scaler, device=device)

    results = pd.DataFrame(
        {
            'Lattice_Index': np.arange(len(predictions)),
            'Predicted_Stiffness': predictions,
            'Actual_Stiffness': ground_truth,
            'Absolute_Error': np.abs(predictions - ground_truth),
            'Percent_Difference': np.abs(predictions - ground_truth) / np.clip(np.abs(ground_truth), a_min=np.finfo(np.float32).eps, a_max=None) * 100.0,
        }
    )
    return results, metrics


In [ ]:
config = TrainingConfig()
set_seed(config.seed)

print(f'Device: {config.device}')
print(f'Batch size: {config.batch_size}')
print(f'Hidden dim: {config.hidden_dim}')
print(f'Epochs: {config.total_epochs}')
print('Feature mode: weighted edges + richer node features + graph-level summary features')
print('Target mode: standardized target y during training, inverse-transformed for evaluation')

In [ ]:
dataset = load_lattice_dataset(train_root)
train_data, val_data, test_data = split_dataset(dataset, seed=config.seed)
feature_scaler = normalize_feature_splits(train_data, val_data, test_data)
target_scaler = normalize_target_splits(train_data, val_data, test_data)
train_loader, val_loader, test_loader = create_data_loaders(
    train_data,
    val_data,
    test_data,
    batch_size=config.batch_size,
)

model = SimpleGNN(
    input_dim=train_data[0].x.shape[1],
    hidden_dim=config.hidden_dim,
    graph_feature_dim=train_data[0].graph_attr.shape[1],
)

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print(f'Test samples: {len(test_data)}')
print(f'Input feature dimension: {train_data[0].x.shape[1]}')
print(f'Graph feature dimension: {train_data[0].graph_attr.shape[1]}')
print(f'Target scaler mean: {float(target_scaler.mean_[0]):.8f}')
print(f'Target scaler scale: {float(target_scaler.scale_[0]):.8f}')
model

In [ ]:
history = train_target_normalized_model(model, train_loader, val_loader, config)
plot_training_history(history, config.epochs_phase1)

In [ ]:
metrics_by_split = {}
split_results = []

for split_name, loader in (("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)):
    predictions, ground_truth, metrics = evaluate_target_normalized_model(model, loader, target_scaler, device=config.device)
    metrics_by_split[split_name] = metrics
    split_results.append((split_name, predictions, ground_truth))

metrics_frame = summarize_metrics(metrics_by_split)
metrics_frame

In [ ]:
plot_prediction_splits(split_results)

In [ ]:
prediction_results, prediction_metrics = predict_on_directory_target_normalized(
    model,
    predict_root,
    feature_scaler,
    target_scaler,
    device=config.device,
)

prediction_summary = pd.Series(prediction_metrics, name='Prediction Set')
display(prediction_results.head())
display(prediction_summary)

saved_dir = save_run_artifacts(
    output_dir,
    model,
    feature_scaler,
    history,
    metrics_by_split,
    prediction_results=prediction_results,
)

target_scaler_payload = {
    'mode': 'standardize_y',
    'mean': float(target_scaler.mean_[0]),
    'scale': float(target_scaler.scale_[0]),
}
with (saved_dir / 'target_scaler.json').open('w', encoding='utf-8') as handle:
    json.dump(target_scaler_payload, handle, indent=2)

saved_files = sorted(path.name for path in saved_dir.iterdir() if path.is_file())
print(f'Saved artifacts to {saved_dir}')
print(f'Latest run pointer: {output_root / "latest_run.txt"}')
print('Saved files:')
for name in saved_files:
    print(f' - {name}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / saved_dir.name
    git_run_dir.mkdir(parents=True, exist_ok=True)

    files_to_push = ['metrics_summary.csv', 'prediction_results.csv', 'target_scaler.json']
    if PUSH_MODEL_TO_GITHUB:
        files_to_push.append('lattice_gnn_model.pt')

    for file_name in files_to_push:
        source_path = saved_dir / file_name
        if source_path.is_file():
            shutil.copy2(source_path, git_run_dir / file_name)

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add target-normalized richer-graph-feature GCN cloud results for {saved_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_richer_graph_features_target_normalized.zip'
    !cd /content && zip -qr gnn_outputs_richer_graph_features_target_normalized.zip gnn_outputs_richer_graph_features_target_normalized
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')